# Pre-processing a .CHA file for use in CE analysis

In [1]:
import os
import sys
import pandas as pd
from tqdm import tqdm

In [3]:
data_path = '../data'
chas_path = os.path.join(data_path, 'chas')
outputs_path = os.path.join(data_path, 'server_ready', 'cha_corpus-coraal.csv')

In [4]:
def grab_all_files(PATH, file_type='.cha'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

# Processing all CHA files

Note: the package used here was developed by Prof. Garber. Cite via:

Garber, L. (2019). CHA file python parser. Zenodo. https://doi.org/10.5281/zenodo.3364020

In [5]:
from shared.CHAFile import *

In [6]:
all_files = grab_all_files(chas_path)
len(all_files), all_files[0]

(271, '../data/chas/CORAAL/ATL/se0-ag1/se0_ag1_f_01.cha')

In [7]:
data = []
for f in all_files:
    
    chacha = ChaFile(f)
    meta_data_pieces = f.replace('.cha', '').split('/')
    
    for line in chacha.getLines():
        line['overlapping_text'] = bool(re.findall(r"(⌋|⌊|⌉|⌈)", line['text']))

        if meta_data_pieces[-2] in ['eng_n', 'eng_s']:
            line['corpus'] = meta_data_pieces[-3] + '-' + meta_data_pieces[-2]
        
        elif meta_data_pieces[-3] in ['instruction', 'CABNC', 'MICASE']:
            line['corpus'] = meta_data_pieces[-3] + '-' + meta_data_pieces[-2]
        
        else:
            line['corpus'] = meta_data_pieces[-2]
        
        line['conversation_id'] = line['corpus'] + '-' + meta_data_pieces[-1]
        
        data += [line.copy()]

data = pd.DataFrame(data)

In [8]:
data.shape

(202766, 12)

In [9]:
data.head()

,document_line_no,utterance_no,speaker,text,bullet,wor,recipient,overlapping_text,corpus,conversation_id,com,exp
0,10,1,PAR,they talking about don't send him to his daddy .,"[315, 2335]",they 315_635 talking 635_715 about 715_83...,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN
1,12,2,PAR,you just needta go file for child support .,"[2675, 4695]",you 2675_2815 just 2815_2995 needta 2995_...,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN
2,14,3,INV,oh man .,NaN,NaN,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN
3,15,4,PAR,bye .,NaN,NaN,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN
4,16,5,PAR,why ?,"[5355, 5855]",why 5355_5855 ?,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN


In [10]:
data['conversation_id'].nunique()

213

### Correcting utterances/removing CLAN specific formatting.

In [11]:
def corrected_text(text, contraction_replacement_nonce='CCOONNTTRRAACCTTIIOONN'):
    res = re.sub(r'\(\(.*?\)\)', ' ', str(text))
    # res = re.sub(r'\[.*?\]', ' ', res)
    
    # find contractions and preserve them . . .
    contractions = list(re.findall(r"\w+'\w+", res))
    for contraction in set(contractions):
        replacement = contraction.replace("'", contraction_replacement_nonce)
        res = res.replace(contraction, replacement)
    res = re.sub(r"(⌋|⌊|⌉|⌈)", '', res)
    res = res.replace(':', '')
    
    # remove numbers in parentheses (times???)
    res = re.sub(r'\(\d\.\d\)', ' ', res)
    
    # remove all other special characters.
    res = re.sub(r'[^\w\s]', ' ', res)
    
    res = re.sub(r'\s+', ' ', res).replace('[', ' ').replace(']', ' ').replace(contraction_replacement_nonce, "'")
    return res.strip()

In [12]:
data['raw_text'] = data['text'].values
data['text'] = [corrected_text(text) for text in tqdm(data['raw_text'].values)]

100%|██████████| 202766/202766 [00:01<00:00, 159368.78it/s]


In [13]:
data[['corpus', 'raw_text', 'text']].head(n=6)

,corpus,raw_text,text
0,se0-ag1,they talking about don't send him to his daddy .,they talking about don't send him to his daddy
1,se0-ag1,you just needta go file for child support .,you just needta go file for child support
2,se0-ag1,oh man .,oh man
3,se0-ag1,bye .,bye
4,se0-ag1,why ?,why
5,se0-ag1,why ?,why


## Create juxtaposed corpus: (x,y) pairs

In [14]:
max_turns_apart = 30

In [15]:
import warnings; warnings.filterwarnings("ignore")

corpus = []
for cid in tqdm(data['conversation_id'].unique()):
    sub = data.loc[data['conversation_id'].isin([cid])]
    sub_index = sub.index.values
    
    for i in sub_index:
        if i != sub_index[-1]:
            
            # speaker vs. other
            next_line_no = ( (sub_index > i) & (~sub['speaker'].isin([sub['speaker'].loc[i]])) ).values.nonzero()[0]
            next_line_no = sub_index[next_line_no][:(max_turns_apart+1)]
            
            # next_line_no = next_line_no[next_line_no <= (i + max_turns_apart)]
            for j,li in enumerate(next_line_no):
                d = data.loc[i].to_dict()
                
                d['next_speaker'] = data['speaker'].loc[li]
                d['next_text'] = data['text'].loc[li]
                d['next_utterance_no'] = data['utterance_no'].loc[li]
                d['next_utterance_delta_no'] = j
                
                corpus += [d]
            
            # speaker vs. self 
            next_line_no = ( (sub_index > i) & (sub['speaker'].isin([sub['speaker'].loc[i]])) ).values.nonzero()[0]
            next_line_no = sub_index[next_line_no][:(max_turns_apart+1)]
            # next_line_no = next_line_no[next_line_no <= (i + max_turns_apart)]
            for j,li in enumerate(next_line_no):
                d = data.loc[i].to_dict()
                
                d['next_speaker'] = data['speaker'].loc[li]
                d['next_text'] = data['text'].loc[li]
                d['next_utterance_no'] = data['utterance_no'].loc[li]
                d['next_utterance_delta_no'] = j
                
                corpus += [d]

100%|██████████| 213/213 [10:22<00:00,  2.92s/it]


In [16]:
data = pd.DataFrame(corpus)
print(data.shape)
data.head()

(11936993, 17)


,document_line_no,utterance_no,speaker,text,bullet,wor,recipient,overlapping_text,corpus,conversation_id,com,exp,raw_text,next_speaker,next_text,next_utterance_no,next_utterance_delta_no
0,10,1,PAR,they talking about don't send him to his daddy,"[315, 2335]",they 315_635 talking 635_715 about 715_83...,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN,they talking about don't send him to his daddy .,INV,oh man,3,0
1,10,1,PAR,they talking about don't send him to his daddy,"[315, 2335]",they 315_635 talking 635_715 about 715_83...,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN,they talking about don't send him to his daddy .,INV,okay what's your name,7,1
2,10,1,PAR,they talking about don't send him to his daddy,"[315, 2335]",they 315_635 talking 635_715 about 715_83...,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN,they talking about don't send him to his daddy .,INV,Personname what,9,2
3,10,1,PAR,they talking about don't send him to his daddy,"[315, 2335]",they 315_635 talking 635_715 about 715_83...,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN,they talking about don't send him to his daddy .,INV,okay and uh are you a male or female,11,3
4,10,1,PAR,they talking about don't send him to his daddy,"[315, 2335]",they 315_635 talking 635_715 about 715_83...,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN,they talking about don't send him to his daddy .,INV,laughs,13,4


In [17]:
rename_columns = dict()
for col in data.columns:
    if col == 'text':
        rename_columns[col] = 'x_utterance'
    elif col == 'next_text':
        rename_columns[col] = 'y_utterance'
    elif 'next_' in col:
        rename_columns[col]  = col.replace('next', 'y')
    else:
        rename_columns[col] = col

In [18]:
data = data.rename(columns=rename_columns)
data.head()

,document_line_no,utterance_no,speaker,x_utterance,bullet,wor,recipient,overlapping_text,corpus,conversation_id,com,exp,raw_text,y_speaker,y_utterance,y_utterance_no,y_utterance_delta_no
0,10,1,PAR,they talking about don't send him to his daddy,"[315, 2335]",they 315_635 talking 635_715 about 715_83...,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN,they talking about don't send him to his daddy .,INV,oh man,3,0
1,10,1,PAR,they talking about don't send him to his daddy,"[315, 2335]",they 315_635 talking 635_715 about 715_83...,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN,they talking about don't send him to his daddy .,INV,okay what's your name,7,1
2,10,1,PAR,they talking about don't send him to his daddy,"[315, 2335]",they 315_635 talking 635_715 about 715_83...,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN,they talking about don't send him to his daddy .,INV,Personname what,9,2
3,10,1,PAR,they talking about don't send him to his daddy,"[315, 2335]",they 315_635 talking 635_715 about 715_83...,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN,they talking about don't send him to his daddy .,INV,okay and uh are you a male or female,11,3
4,10,1,PAR,they talking about don't send him to his daddy,"[315, 2335]",they 315_635 talking 635_715 about 715_83...,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN,they talking about don't send him to his daddy .,INV,laughs,13,4


In [19]:
data = data.rename(columns={'speaker':'x_speaker','utterance_no': 'x_turn_id', 'y_utterance_no': 'y_turn_id'})
data.head()

,document_line_no,x_turn_id,x_speaker,x_utterance,bullet,wor,recipient,overlapping_text,corpus,conversation_id,com,exp,raw_text,y_speaker,y_utterance,y_turn_id,y_utterance_delta_no
0,10,1,PAR,they talking about don't send him to his daddy,"[315, 2335]",they 315_635 talking 635_715 about 715_83...,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN,they talking about don't send him to his daddy .,INV,oh man,3,0
1,10,1,PAR,they talking about don't send him to his daddy,"[315, 2335]",they 315_635 talking 635_715 about 715_83...,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN,they talking about don't send him to his daddy .,INV,okay what's your name,7,1
2,10,1,PAR,they talking about don't send him to his daddy,"[315, 2335]",they 315_635 talking 635_715 about 715_83...,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN,they talking about don't send him to his daddy .,INV,Personname what,9,2
3,10,1,PAR,they talking about don't send him to his daddy,"[315, 2335]",they 315_635 talking 635_715 about 715_83...,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN,they talking about don't send him to his daddy .,INV,okay and uh are you a male or female,11,3
4,10,1,PAR,they talking about don't send him to his daddy,"[315, 2335]",they 315_635 talking 635_715 about 715_83...,ADULT,False,se0-ag1,se0-ag1-se0_ag1_f_01,NaN,NaN,they talking about don't send him to his daddy .,INV,laughs,13,4


In [20]:
data['x_speaker'] = data['corpus'].astype(str) + '-' + data['conversation_id'].astype(str) + '-' + data['x_speaker'].astype(str)

data['y_speaker'] = data['corpus'].astype(str) + '-' + data['conversation_id'].astype(str) + '-' + data['y_speaker'].astype(str)

In [21]:
data['self'] = data['x_speaker'] == data['y_speaker']
data = data.sort_values(by=['corpus', 'conversation_id', 'x_turn_id', 'self', 'y_turn_id'])
data.index = range(len(data))

In [22]:
data[['corpus', 'x_utterance', 'y_utterance']].isna().mean()

corpus         0.0
x_utterance    0.0
y_utterance    0.0
dtype: float64

In [23]:
# # corpus sanity check . . . make sure that conversation_ids are all unique 
# k = data[['conversation_id', 'corpus']].drop_duplicates()
# k['conversation_id'].nunique(), k['conversation_id'].nunique()/len(k)

In [24]:
data[['corpus', 'x_utterance', 'x_speaker', 'y_speaker', 'y_utterance', 'x_turn_id', 'y_turn_id']].head()

,corpus,x_utterance,x_speaker,y_speaker,y_utterance,x_turn_id,y_turn_id
0,se0-ag1,they talking about don't send him to his daddy,se0-ag1-se0-ag1-se0_ag1_f_01-PAR,se0-ag1-se0-ag1-se0_ag1_f_01-INV,oh man,1,3
1,se0-ag1,they talking about don't send him to his daddy,se0-ag1-se0-ag1-se0_ag1_f_01-PAR,se0-ag1-se0-ag1-se0_ag1_f_01-INV,okay what's your name,1,7
2,se0-ag1,they talking about don't send him to his daddy,se0-ag1-se0-ag1-se0_ag1_f_01-PAR,se0-ag1-se0-ag1-se0_ag1_f_01-INV,Personname what,1,9
3,se0-ag1,they talking about don't send him to his daddy,se0-ag1-se0-ag1-se0_ag1_f_01-PAR,se0-ag1-se0-ag1-se0_ag1_f_01-INV,okay and uh are you a male or female,1,11
4,se0-ag1,they talking about don't send him to his daddy,se0-ag1-se0-ag1-se0_ag1_f_01-PAR,se0-ag1-se0-ag1-se0_ag1_f_01-INV,laughs,1,13


In [25]:
data.shape

(11936993, 18)

## Save outputs for server operations.

In [26]:
data.to_csv(outputs_path, index=False, encoding='utf-8')